[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/buddywhitman/57e1f6b40953a2eec1cb1fa176c93fee/investigation_notebook.ipynb)

# Data-Driven Detection of Anomalous Pollution Spikes Using Python
### Course Project: ELE 4449 - Introduction to Data Science
**Department of Electrical Engineering**  
**Authors:** Pulkit Kumar, Vijayalakshmi AL, Anant Prasad, Rama Sridhrut  
**Station:** CPCB CAAQM Station: Alipur, Delhi (`site_5024`) &bull; **Period:** November 1–30, 2024  
**Weather Reanalysis:** European Centre for Medium-Range Weather Forecasts (ECMWF) ERA5  

---
### Overview & Motivation
This computational notebook provides an end-to-end, interpretable time-series anomaly screening framework for urban continuous ambient air quality monitoring. It tests whether extreme pollution spikes are consistent with atmospheric dispersion and multi-pollutant co-emission, or represent potential instrument artifacts.

> **Cloud Execution Guarantee:** This notebook runs identically locally or in Google Colab with zero configuration. When running in Colab, it automatically streams the pre-processed observation dataset from the authenticated public Gist repository.

## 1. Environment Setup & Seamless Cloud Data Loading
We dynamically resolve the dataset location: if running locally, it reads from the workspace; if running in Google Colab or a cloud container, it downloads the verified dataset directly from the raw Gist URL.

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# Configure Plotly rendering for seamless Colab compatibility
try:
    import google.colab
    pio.renderers.default = 'colab'
except ImportError:
    pio.renderers.default = 'notebook_connected'

# Dynamic Data Source Resolution: Local Path or Cloud Gist Raw URL
local_paths = [
    Path('outputs/hourly_scored_observations.csv'),
    Path('mini-project/outputs/hourly_scored_observations.csv'),
    Path('hourly_scored_observations.csv')
]

cloud_url = 'https://gist.githubusercontent.com/buddywhitman/57e1f6b40953a2eec1cb1fa176c93fee/raw/hourly_scored_observations.csv'

data_source = None
for p in local_paths:
    if p.exists():
        data_source = str(p)
        break

if data_source is not None:
    print(f'[Local Runtime] Loaded dataset from {data_source}')
    df = pd.read_csv(data_source)
else:
    print(f'[Google Colab Runtime] Streaming dataset directly from Cloud Gist: {cloud_url}')
    df = pd.read_csv(cloud_url)

df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'[Runtime Info] Successfully resolved dataset: {len(df)} observation hours across {len(df.columns)} features.')
df[['timestamp', 'pm25', 'no2', 'wind_speed', 'pollution_index', 'status']].head()

[Runtime Info] Successfully resolved dataset: 701 observation hours across 25 features.


,timestamp,pm25,no2,wind_speed,pollution_index,status
0,2024-11-01 00:00:00,351.0,38.43,11.3,103.498970,Normal
1,2024-11-01 01:00:00,439.0,38.30,11.3,111.528982,Normal
2,2024-11-01 02:00:00,407.0,35.00,11.3,107.109841,Normal
3,2024-11-01 03:00:00,383.0,34.95,10.4,104.881565,Normal
4,2024-11-01 04:00:00,403.0,34.00,10.3,106.294326,Normal


## 2. Statistical Baselines & Robust Non-Parametric Estimators
Urban pollution series exhibit heavy positive skewness ($\gamma_1 > 2.5$) and extreme leptokurtic Pareto-like tails. Gaussian z-score standardization ($z = \frac{x - \mu}{\sigma}$) breaks down (0% breakdown point) because acute multi-day smog episodes inflate $\mu$ and $\sigma$, desensitizing anomaly detection. We enforce non-parametric Median (50% breakdown) and Interquartile Range (IQR, 25% breakdown) estimators.

In [2]:
summary_stats = pd.DataFrame({
    'Metric': ['Median', 'IQR (Q3 - Q1)', 'Skewness', 'Minimum', 'Maximum', '95th Percentile'],
    'PM2.5 (ug/m3)': [
        df['pm25'].median(),
        df['pm25'].quantile(0.75) - df['pm25'].quantile(0.25),
        df['pm25'].skew(),
        df['pm25'].min(),
        df['pm25'].max(),
        df['pm25'].quantile(0.95)
    ],
    'NO2 (ug/m3)': [
        df['no2'].median(),
        df['no2'].quantile(0.75) - df['no2'].quantile(0.25),
        df['no2'].skew(),
        df['no2'].min(),
        df['no2'].max(),
        df['no2'].quantile(0.95)
    ],
    'Wind Speed (km/h)': [
        df['wind_speed'].median(),
        df['wind_speed'].quantile(0.75) - df['wind_speed'].quantile(0.25),
        df['wind_speed'].skew(),
        df['wind_speed'].min(),
        df['wind_speed'].max(),
        df['wind_speed'].quantile(0.95)
    ]
}).round(2)
summary_stats

Metric,PM2.5 (ug/m3),NO2 (ug/m3),Wind Speed (km/h)
Median,249.00,51.55,5.80
IQR (Q3 - Q1),136.00,27.91,3.20
Skewness,1.44,0.90,0.39
Minimum,41.00,18.98,0.40
Maximum,922.00,160.57,16.70
95th Percentile,619.00,97.08,9.80


## 3. Rolling Baselines & Target Leakage Prevention
A 6-hour backward-looking rolling window is applied. Crucially, `.shift(1)` is strictly enforced prior to `.rolling(6)` to decouple observation $t$ from its own baseline, preventing the spike from dampening its own relative jump metric.

In [3]:
# Rolling median and rolling MAD with .shift(1)
df['calc_roll_med'] = df['pollution_index'].shift(1).rolling(6, min_periods=3).median()
df['calc_roll_mad'] = df['pollution_index'].shift(1).rolling(6, min_periods=3).apply(
    lambda x: np.median(np.abs(x - np.median(x))), raw=True
)

# Relative jump metric on composite index
delta = (df['pollution_index'] - df['calc_roll_med']).abs()
df['calc_rel_jump'] = delta / (df['calc_roll_med'] + 1)

print("Max relative jump observed on rescaled index:", round(df['calc_rel_jump'].max(), 3))
print("Quantiles of relative jump:")
print(df['calc_rel_jump'].quantile([0.5, 0.75, 0.9, 0.95, 0.99, 1.0]).round(3))

Max relative jump observed on rescaled index: 0.629
Quantiles of relative jump:
0.50    0.082
0.75    0.164
0.90    0.262
0.95    0.333
0.99    0.485
1.00    0.629


## 4. Multi-Criteria Anomaly Classification Breakdown
The transparent rule synthesizes 4 criteria: Extreme Flag (35 pts), Large Jump Flag (35 pts), Unsupported Spike Flag (20 pts), and Disagreement Flag (10 pts). The resulting score partitions data into Normal ($0-30$), Watch ($31-65$), and Suspicious ($>65$).

In [4]:
status_counts = df['status'].value_counts()
print("Classification Breakdown (701 observation hours):")
print(status_counts)

watch_df = df[df['status'] == 'Watch']
print(f"\nWatch rows score value counts:")
print(watch_df['suspicion_score'].value_counts())

Classification Breakdown:
status
Normal    666
Watch      35

Watch rows score value counts:
suspicion_score
45    32
35     3


## 5. Interactive Visual Screening (Plotly Timeline)
Temporal progression of the Pollution Severity Index proxy with Watch anomaly markers highlighted in amber.

In [5]:
fig_timeline = px.line(
    df, x="timestamp", y="pollution_index",
    title="Alipur, Delhi: Hourly Pollution Severity Index with Anomaly Flags (November 2024)",
    labels={"timestamp": "Time (IST)", "pollution_index": "Pollution Severity Index Proxy"}
)

if not watch_df.empty:
    fig_timeline.add_trace(go.Scatter(
        x=watch_df["timestamp"], y=watch_df["pollution_index"],
        mode="markers", name="Watch hours (Score 35-45)",
        marker=dict(color="#d97706", size=8, symbol="circle-open", line=dict(width=2)),
        text=watch_df["suspicion_score"],
        hovertemplate="%{x}<br>Index: %{y:.1f}<br>Score: %{text}<extra></extra>"
    ))

fig_timeline.update_layout(template="plotly_white", height=480)
fig_timeline.show()

## 6. Decoupling Pollution Severity from Sensor Suspicion
Scatter plot demonstrating that high pollution severity does not automatically imply sensor suspicion.

In [6]:
df_plot = df.copy()
df_plot["marker_size"] = df_plot["relative_jump"].fillna(0).clip(lower=0.05)

fig_scatter = px.scatter(
    df_plot, x="pollution_index", y="suspicion_score", color="status",
    size="marker_size", size_max=16,
    hover_data=["timestamp", "pm25", "no2", "wind_speed", "relative_jump"],
    color_discrete_map={"Normal": "#10b981", "Watch": "#f59e0b", "Suspicious": "#ef4444"},
    title="Decoupling Severity from Suspicion: Pollution Index vs. Suspicion Score",
    labels={"pollution_index": "Pollution Severity Index Proxy", "suspicion_score": "Suspicion Score"}
)
fig_scatter.update_layout(template="plotly_white", height=480)
fig_scatter.show()

## 7. Interactive Threshold Sandbox: Proving Affine Metric Compression
Mathematical diagnostic: why relative percentage thresholds compress under affine shifts ($y = 100 + 25z$), and what happens when the threshold is recalibrated.

In [7]:
def run_threshold_experiment(jump_threshold=1.0, extreme_pct=0.95):
    test_df = df.copy()
    q_val = test_df['pollution_index'].quantile(extreme_pct)
    
    ext_flag = (test_df['pollution_index'] > q_val).astype(int)
    jump_flag = (test_df['relative_jump'] > jump_threshold).astype(int)
    low_wind = (test_df['wind_speed'] < 0.6 * test_df['wind_speed'].median()).astype(int)
    unsupp = (jump_flag & (1 - low_wind)).astype(int)
    disag = ((test_df['pm25_severity'] - test_df['no2_severity']).abs() > 2.0).astype(int)
    
    score = 35 * ext_flag + 35 * jump_flag + 20 * unsupp + 10 * disag
    status = pd.cut(score, bins=[-1, 30, 65, np.inf], labels=['Normal', 'Watch', 'Suspicious'])
    
    counts = status.value_counts()
    print(f"--- Experiment: Jump Threshold = {jump_threshold}, Extreme Percentile = {int(extreme_pct*100)}th ---")
    print(counts.to_string())
    return counts

# Standard Project Rule (Threshold = 1.0) -> Suspicious = 0
run_threshold_experiment(jump_threshold=1.0, extreme_pct=0.95)

# Recalibrated Threshold (Threshold = 0.35) -> Shows active Suspicious classifications
run_threshold_experiment(jump_threshold=0.35, extreme_pct=0.95)

--- Experiment: Jump Threshold = 1.0, Extreme Percentile = 95th ---
Normal        666
Watch          35
Suspicious      0

--- Experiment: Jump Threshold = 0.35, Extreme Percentile = 95th ---
Normal        639
Watch          59
Suspicious      3


## 8. Forensic Case Study: November 12–14 Episode
Detailed inspection of the month's maximum peak (where PM2.5 escalated from 150 to 887 ug/m3). The framework correctly classified it as Watch (Score 45) rather than Suspicious, because concurrent wind deceleration down to 5.1 km/h and multi-hour persistence confirmed a genuine boundary-layer inversion.

In [8]:
case = df[(df['timestamp'] >= '2024-11-12 18:00:00') & (df['timestamp'] <= '2024-11-13 12:00:00')]
case[['timestamp', 'pm25', 'no2', 'wind_speed', 'pollution_index', 'relative_jump', 'suspicion_score', 'status']].head(10)

timestamp,pm25,no2,wind_speed,pollution_index,relative_jump,suspicion_score,status
2024-11-12 18:00:00,119.0,48.80,11.4,86.819833,0.175573,0,Normal
2024-11-12 19:00:00,150.0,49.90,10.3,90.161753,0.220308,0,Normal
2024-11-12 20:00:00,233.0,51.22,8.9,98.381615,0.300182,0,Normal
2024-11-12 21:00:00,459.0,58.32,8.9,122.333538,0.466285,0,Normal
2024-11-12 22:00:00,572.0,66.65,8.1,136.450309,0.535916,35,Watch
2024-11-12 23:00:00,781.0,63.48,7.3,154.240126,0.629447,45,Watch
2024-11-13 00:00:00,631.0,62.15,5.1,139.857696,0.264913,45,Watch
2024-11-13 01:00:00,713.0,51.35,5.0,142.557485,0.100969,45,Watch
2024-11-13 02:00:00,699.0,47.50,5.6,139.546428,0.010006,45,Watch
2024-11-13 03:00:00,887.0,63.08,6.0,163.803626,0.171295,45,Watch
